# LeetCode 4092 — Binary Digit Score bằng Python

Link đề: https://leetcode.com/problems/binary-digit-score/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Math (đếm bit 1 / popcount)

Skeleton LeetCode:

```python
class Solution:
    def calculateScore(self, a: int, b: int) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc (Weekly Contest 525, Q1); mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force viết độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số bit `1` (set bits) của `a` và `b` trong dạng nhị phân, gọi là `s1`, `s2`, rồi tính điểm theo **hai trường hợp** (`s1 >= s2` thì nhân, ngược lại thì chia **nguyên**).
2. Biết **biểu diễn nhị phân** của một số nguyên và cách đếm số bit `1` bằng nhiều cách: chuỗi `bin()`, dịch bit `>> 1` kèm `& 1`, thủ thuật **Brian Kernighan** `x &= x - 1`, và `int.bit_count()`.
3. Phân biệt **`/`** (chia thực, ra `float`) với **`//`** (chia nguyên) và hiểu vì sao đề dùng "integer division".
4. Nhận ra vì sao **không bao giờ chia cho `0`**: `a >= 1` nên `s1 >= 1`.
5. Tránh các lỗi kinh điển: dùng `/` thay `//`, đếm số chữ số nhị phân (độ dài) thay vì số bit `1`, đảo thứ tự nhánh `>=`/`<`, quên `0b` khi đếm bằng chuỗi.

## 2. Đề bài & ví dụ

Cho hai số nguyên `a` và `b`. Gọi `s1`, `s2` lần lượt là **số bit 1** (set bits) trong biểu diễn nhị phân của `a` và `b`.

**Điểm (score)** được định nghĩa:

- Nếu `s1 >= s2`: điểm là `s1 * s2`.
- Ngược lại (`s1 < s2`): điểm là `s2 / s1` theo **phép chia nguyên**.

Trả về điểm của `a` và `b`.

**Ví dụ 1**

- Input: `a = 14, b = 6`
- Output: `6`
- Giải thích: `14` là `"1110"` nên `s1 = 3`; `6` là `"110"` nên `s2 = 2`. Vì `s1 > s2`, điểm là `3 * 2 = 6`.

**Ví dụ 2**

- Input: `a = 5, b = 15`
- Output: `2`
- Giải thích: `5` là `"101"` nên `s1 = 2`; `15` là `"1111"` nên `s2 = 4`. Vì `s1 < s2`, điểm là `4 / 2 = 2`.

**Ví dụ 3**

- Input: `a = 10, b = 12`
- Output: `4`
- Giải thích: `10` là `"1010"` nên `s1 = 2`; `12` là `"1100"` nên `s2 = 2`. Vì `s1 == s2`, điểm là `2 * 2 = 4`.

**Ràng buộc:** `1 <= a, b <= 100`.

In [1]:
from typing import List
import random

examples = [
    (14, 6, 6),
    (5, 15, 2),
    (10, 12, 4),
]
for a, b, expected in examples:
    print(f"a={a}, b={b} -> kỳ vọng {expected}")

a=14, b=6 -> kỳ vọng 6
a=5, b=15 -> kỳ vọng 2
a=10, b=12 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Hệ nhị phân và hàm `bin()`

Máy tính lưu số bằng các **bit** `0` / `1`. Mỗi vị trí bit có trọng số là lũy thừa của `2` (từ phải sang trái: `1, 2, 4, 8, ...`). Ví dụ `14 = 8 + 4 + 2 = 1110` (nhị phân). Hàm `bin(x)` trả chuỗi có tiền tố `0b`, ví dụ `bin(14) == '0b1110'`. Số bit `1` còn gọi là **popcount** hoặc **Hamming weight**.

In [2]:
for x in (14, 6, 5, 15, 10, 12, 1, 100):
    print(f"{x:>3} = {bin(x):>10}  (bỏ '0b': {bin(x)[2:]})")

 14 =     0b1110  (bỏ '0b': 1110)
  6 =      0b110  (bỏ '0b': 110)
  5 =      0b101  (bỏ '0b': 101)
 15 =     0b1111  (bỏ '0b': 1111)
 10 =     0b1010  (bỏ '0b': 1010)
 12 =     0b1100  (bỏ '0b': 1100)
  1 =        0b1  (bỏ '0b': 1)
100 =  0b1100100  (bỏ '0b': 1100100)


### 3.2. Đếm ký tự `'1'` trong chuỗi: `str.count`

`"1110".count("1")` trả số lần `'1'` xuất hiện. Hãy cẩn thận chuỗi `bin(x)` có `'0b'` ở đầu; may mắn ký tự `'b'` và `'0'` không phải `'1'` nên `bin(x).count('1')` vẫn đúng. Nhưng **đếm `'0'`** thì sẽ sai vì tiền tố `0b` có một `'0'`.

In [3]:
s = bin(14)
print(s, s.count("1"))         # 0b1110 3
print(s.count("0"))            # 2 (nhưng chỉ 1 chữ số 0 thật!) -> bẫy
print(len(s) - 2)              # 4 = số chữ số nhị phân (độ dài), KHÔNG phải số bit 1

0b1110 3
2
4


### 3.3. Toán tử bit: `&`, `>>`

- `x & 1` lấy **bit cuối** (bit thấp nhất): `14 & 1 = 0` (vì `1110` kết thúc bằng `0`), `13 & 1 = 1`.
- `x >> 1` **dịch phải một bit** = chia nguyên cho `2`: bỏ bit cuối. `14 >> 1 = 7` (`1110 -> 111`).
- Lặp: nhìn bit cuối, rồi dịch phải, cho tới khi `x = 0` thì ta đã đi qua mọi bit.

In [4]:
x = 14
print(bin(x), x & 1, bin(x >> 1), (x >> 1) & 1)
x = 13
print(bin(x), x & 1, bin(x >> 1), (x >> 1) & 1)

0b1110 0 0b111 1
0b1101 1 0b110 0


### 3.4. Thủ thuật `x & (x - 1)` (Brian Kernighan)

Trừ `1` khỏi `x` làm **bit `1` thấp nhất** của `x` thành `0` và mọi bit `0` bên phải nó thành `1`. Vì vậy `x & (x - 1)` **xóa đúng một bit `1` thấp nhất**. Ví dụ `x = 12 = 1100`, `x - 1 = 1011`, `x & (x-1) = 1000 = 8`. Lặp cho tới khi `x = 0`; số lần lặp chính là số bit `1`.

In [5]:
x = 0b1100
while x:
    print(f"x = {x:04b}, x-1 = {x-1:04b}, x & (x-1) = {x & (x-1):04b}")
    x &= x - 1

x = 1100, x-1 = 1011, x & (x-1) = 1000
x = 1000, x-1 = 0111, x & (x-1) = 0000


### 3.5. Phép chia: `/` và `//`

Trong Python 3, `7 / 2 = 3.5` (chia thực, kiểu `float`), còn `7 // 2 = 3` (chia nguyên, làm tròn xuống). Đề nói "integer division" nên dùng `//`. Với số dương không có sự khác biệt về dấu, nhưng `/` trả `float` (ví dụ `4 / 2 = 2.0`) sẽ khiến đáp án có kiểu sai và có thể sai giá trị khi chia không hết.

In [6]:
print(4 / 2, 4 // 2)      # 2.0 2
print(5 / 2, 5 // 2)      # 2.5 2
print(type(4 / 2), type(4 // 2))

2.0 2
2.5 2
<class 'float'> <class 'int'>


### 3.6. `int.bit_count()` (Python 3.10+)

Python 3.10 thêm sẵn `x.bit_count()` trả số bit `1`. Nếu máy chấm dùng Python cũ hơn, dùng `bin(x).count('1')`. Ta sẽ thử cả hai và so khớp.

In [7]:
import sys
print(sys.version_info[:2])
print((14).bit_count() if hasattr(int, "bit_count") else bin(14).count("1"))   # 3
print(all((x).bit_count() == bin(x).count("1") for x in range(1, 101)))        # True

(3, 13)
3
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài chia làm hai bước hoàn toàn độc lập:

1. **Đếm bit `1`** của `a` và `b`.
2. **Áp công thức** theo so sánh `s1` với `s2`.

Tính tay ví dụ 2 (`a = 5, b = 15`):

- `5 = 4 + 1 = 101` → hai bit `1` → `s1 = 2`.
- `15 = 8 + 4 + 2 + 1 = 1111` → bốn bit `1` → `s2 = 4`.
- `s1 < s2` nên điểm là `s2 // s1 = 4 // 2 = 2`.

Nhận xét: nhánh chia an toàn vì ta chỉ chia khi `s1 < s2`, và `s1 >= 1` (ràng buộc `a >= 1` nghĩa là `a` có ít nhất một bit `1`). Nếu `a` có thể bằng `0` thì `s1 = 0` và phép chia sẽ lỗi — đây là lý do phải đọc kỹ ràng buộc.

## 5. Cách 1 — Brute Force / baseline: đổi sang chuỗi nhị phân rồi đếm `'1'`

**Ý tưởng.** Làm gần như "đọc đề": `bin(a)[2:]` là chuỗi nhị phân; đếm từng ký tự bằng vòng lặp (hoặc `.count`). Sau đó áp công thức.

In [8]:
def popcount_string(x):
    s = bin(x)[2:]                 # bỏ tiền tố '0b'
    count = 0
    for ch in s:
        if ch == "1":
            count += 1
    return count


def score_string(a, b):
    s1, s2 = popcount_string(a), popcount_string(b)
    if s1 >= s2:
        return s1 * s2
    return s2 // s1


for a, b, expected in examples:
    print(a, b, "->", score_string(a, b), "(kỳ vọng", expected, ")")

14 6 -> 6 (kỳ vọng 6 )
5 15 -> 2 (kỳ vọng 2 )
10 12 -> 4 (kỳ vọng 4 )


In [9]:
# Quan sát từng bước của ví dụ 1
a, b = 14, 6
print("bin(a) =", bin(a)[2:], "-> s1 =", popcount_string(a))
print("bin(b) =", bin(b)[2:], "-> s2 =", popcount_string(b))
s1, s2 = popcount_string(a), popcount_string(b)
print("s1 >= s2 ?", s1 >= s2, "-> điểm =", s1 * s2 if s1 >= s2 else s2 // s1)

bin(a) = 1110 -> s1 = 3
bin(b) = 110 -> s2 = 2
s1 >= s2 ? True -> điểm = 6


**Độ phức tạp (Cách 1).** Gọi `L` là số chữ số nhị phân của số lớn hơn (tối đa `7` vì `100 < 128`). Tạo chuỗi và đếm: **Time `O(L)`**, **Space `O(L)`** cho chuỗi. Với `a, b <= 100` thì là hằng số.

**Nút thắt.** Phải tạo chuỗi trung gian chỉ để đếm; ta có thể đếm trực tiếp trên số bằng phép bit.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát:** việc đếm bit `1` của một số `x` có thể làm **không cần chuỗi**:

- Cách bit cuối: lặp `count += x & 1; x >>= 1` — duyệt hết mọi vị trí bit (khoảng `log2 x` lần).
- Cách Kernighan: lặp `x &= x - 1; count += 1` — mỗi vòng **xóa đúng một bit `1`**, nên số vòng bằng chính số bit `1` (không phụ thuộc số bit `0`).
- Hoặc dùng hàm dựng sẵn `bit_count()`.

Phần "điểm" chỉ là một `if/else` đơn giản sau khi có `s1`, `s2`.

## 7. Các cách tiếp theo

### Cách 2 — Dịch bit: `x & 1` rồi `x >>= 1`

**Giải thích.** Mỗi vòng ta cộng bit cuối (`0` hoặc `1`) vào đếm, rồi dịch phải một bit; dừng khi `x == 0`.

In [10]:
def popcount_shift(x):
    count = 0
    while x:
        count += x & 1       # bit cuối là 0 hoặc 1
        x >>= 1              # bỏ bit cuối
    return count


def score_shift(a, b):
    s1, s2 = popcount_shift(a), popcount_shift(b)
    return s1 * s2 if s1 >= s2 else s2 // s1


for a, b, expected in examples:
    print(a, b, "->", score_shift(a, b), "(kỳ vọng", expected, ")")

14 6 -> 6 (kỳ vọng 6 )
5 15 -> 2 (kỳ vọng 2 )
10 12 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 2:** **Time `O(log x)`** (số vòng bằng số chữ số nhị phân), **Space `O(1)`**.

### Cách 3 — Brian Kernighan: `x &= x - 1`

**Giải thích.** Mỗi vòng xóa đúng một bit `1` thấp nhất, đếm số lần xóa đến khi `x = 0`. Số vòng = số bit `1`.

In [11]:
def popcount_kernighan(x):
    count = 0
    while x:
        x &= x - 1           # xóa bit 1 thấp nhất
        count += 1
    return count


def score_kernighan(a, b):
    s1, s2 = popcount_kernighan(a), popcount_kernighan(b)
    return s1 * s2 if s1 >= s2 else s2 // s1


for a, b, expected in examples:
    print(a, b, "->", score_kernighan(a, b), "(kỳ vọng", expected, ")")

14 6 -> 6 (kỳ vọng 6 )
5 15 -> 2 (kỳ vọng 2 )
10 12 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 3:** **Time `O(s)`** với `s` là số bit `1` (tốt hơn Cách 2 khi nhiều bit `0`), **Space `O(1)`**.

### Cách 4 — Hàm dựng sẵn `int.bit_count()`

**Giải thích.** `x.bit_count()` (Python 3.10+) làm đúng việc đếm. Bản nộp trên LeetCode thường dùng cách này vì ngắn nhất; nếu sợ Python cũ, dùng `bin(x).count("1")`.

In [12]:
def score_builtin(a, b):
    s1, s2 = a.bit_count(), b.bit_count()
    return s1 * s2 if s1 >= s2 else s2 // s1


for a, b, expected in examples:
    print(a, b, "->", score_builtin(a, b), "(kỳ vọng", expected, ")")

14 6 -> 6 (kỳ vọng 6 )
5 15 -> 2 (kỳ vọng 2 )
10 12 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 4:** **Time `O(1)`** coi như hằng với số máy (hoặc `O(log x)` theo độ dài bit), **Space `O(1)`**.

### Cách 5 — Bảng tính sẵn bằng quy hoạch động nhỏ

**Giải thích.** Vì `a, b <= 100`, có thể lập bảng `cnt[0..100]` bằng công thức `cnt[i] = cnt[i >> 1] + (i & 1)` (số bit `1` của `i` = số bit `1` của `i` bỏ bit cuối, cộng bit cuối). Sau đó mỗi truy vấn là tra bảng. Cách này đáng nhớ khi phải trả lời **rất nhiều** truy vấn popcount.

In [13]:
cnt = [0] * 101
for i in range(1, 101):
    cnt[i] = cnt[i >> 1] + (i & 1)

print(cnt[:16])                       # [0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4]


def score_table(a, b):
    s1, s2 = cnt[a], cnt[b]
    return s1 * s2 if s1 >= s2 else s2 // s1


for a, b, expected in examples:
    print(a, b, "->", score_table(a, b), "(kỳ vọng", expected, ")")

[0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4]
14 6 -> 6 (kỳ vọng 6 )
5 15 -> 2 (kỳ vọng 2 )
10 12 -> 4 (kỳ vọng 4 )


**Độ phức tạp Cách 5:** dựng bảng **Time `O(M)`** với `M = 100` (một lần), mỗi truy vấn **`O(1)`**; **Space `O(M)`**.

> Cách 4 (hoặc Cách 3 nếu muốn tự viết) là bản nộp gọn nhất. Cách 1 dễ hiểu nhất cho người mới.

## 8. TRACE từng bước

Ta theo dõi Kernighan cho `x = 14`, rồi toàn bộ quá trình tính điểm cho ví dụ 1 và ví dụ 2.

In [14]:
def trace_kernighan(x):
    print(f"popcount của {x} = {x:b}")
    count = 0
    while x:
        new_x = x & (x - 1)
        print(f"  x={x:>3} ({x:07b}) x-1=({x-1:07b}) x&(x-1)=({new_x:07b})")
        x = new_x
        count += 1
    print("  -> số bit 1 =", count)
    return count


def trace_score(a, b):
    s1 = trace_kernighan(a)
    s2 = trace_kernighan(b)
    if s1 >= s2:
        print(f"s1={s1} >= s2={s2}: điểm = s1 * s2 = {s1 * s2}")
        return s1 * s2
    print(f"s1={s1} < s2={s2}: điểm = s2 // s1 = {s2 // s1}")
    return s2 // s1


trace_score(14, 6)
print()
trace_score(5, 15)

popcount của 14 = 1110
  x= 14 (0001110) x-1=(0001101) x&(x-1)=(0001100)
  x= 12 (0001100) x-1=(0001011) x&(x-1)=(0001000)
  x=  8 (0001000) x-1=(0000111) x&(x-1)=(0000000)
  -> số bit 1 = 3
popcount của 6 = 110
  x=  6 (0000110) x-1=(0000101) x&(x-1)=(0000100)
  x=  4 (0000100) x-1=(0000011) x&(x-1)=(0000000)
  -> số bit 1 = 2
s1=3 >= s2=2: điểm = s1 * s2 = 6

popcount của 5 = 101
  x=  5 (0000101) x-1=(0000100) x&(x-1)=(0000100)
  x=  4 (0000100) x-1=(0000011) x&(x-1)=(0000000)
  -> số bit 1 = 2
popcount của 15 = 1111
  x= 15 (0001111) x-1=(0001110) x&(x-1)=(0001110)
  x= 14 (0001110) x-1=(0001101) x&(x-1)=(0001100)
  x= 12 (0001100) x-1=(0001011) x&(x-1)=(0001000)
  x=  8 (0001000) x-1=(0000111) x&(x-1)=(0000000)
  -> số bit 1 = 4
s1=2 < s2=4: điểm = s2 // s1 = 2


2

Bảng tóm tắt (ví dụ 2, `a = 5`, `b = 15`):

| số | nhị phân | các bước `x &= x-1` | số bit 1 |
|---|---|---|---|
| 5 | `101` | `101 -> 100 -> 0` | 2 |
| 15 | `1111` | `1111 -> 1110 -> 1100 -> 1000 -> 0` | 4 |

`s1 = 2 < s2 = 4` → `4 // 2 = 2`.

## 9. Code nộp LeetCode

Bản nộp dùng `bit_count()` (Cách 4). Nếu muốn an toàn với Python cũ, thay bằng `bin(a).count("1")`.

In [15]:
class Solution:
    def calculateScore(self, a: int, b: int) -> int:
        s1 = a.bit_count()
        s2 = b.bit_count()
        return s1 * s2 if s1 >= s2 else s2 // s1


sol = Solution()
for a, b, expected in examples:
    assert sol.calculateScore(a, b) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Đếm bit đúng:** `bit_count` (và hai cách tự viết) đếm đúng số vị trí có bit `1`: Cách 2 xét từng bit từ thấp lên cao qua `x & 1` và `>>= 1`; Cách 3 mỗi vòng xóa **đúng một** bit `1` nên số vòng bằng số bit `1`.
- **Công thức khớp đề:** nhánh `s1 >= s2` trả `s1 * s2`, nhánh còn lại (`s1 < s2`) trả `s2 // s1` — chính là hai dòng của định nghĩa điểm. Hai nhánh phủ kín mọi trường hợp và loại trừ nhau.
- **Không chia cho 0:** nhánh chia chỉ chạy khi `s1 < s2`; hơn nữa `a >= 1` nên `s1 >= 1`.
- **Chia nguyên:** dùng `//` đúng "integer division"; vì `s1, s2` là số nguyên dương nên `//` là phần nguyên của thương.

## 11. Phân tích Time / Space Complexity

Gọi `B` là số bit của số lớn nhất (`B <= 7` vì `a, b <= 100`).

- **Cách 1 (chuỗi):** Time `O(B)`, Space `O(B)`.
- **Cách 2 (dịch bit):** Time `O(B)`, Space `O(1)`.
- **Cách 3 (Kernighan):** Time `O(s)` với `s` là số bit `1` (`s <= B`), Space `O(1)`.
- **Cách 4 (`bit_count`):** Time `O(1)` thực tế (một lệnh phần cứng popcount cho số nhỏ), Space `O(1)`.
- **Cách 5 (bảng):** dựng `O(M)` một lần, tra `O(1)`; Space `O(M)`.

Vì `a, b <= 100`, mọi cách đều là **hằng số**; ta vẫn viết theo `B` để thấy độ lớn tăng theo số bit khi đề mở rộng. Không có khác biệt average/worst đáng kể.

## 12. Bộ test

Ta viết `oracle` brute force độc lập: dùng định nghĩa **trọng số lũy thừa của 2** (không dùng `bin`/`count`/`bit_count`), rồi đối chiếu mọi cách.

In [16]:
def oracle(a, b):
    """Brute force độc lập: tách số thành tổng các lũy thừa của 2 bằng phép trừ."""
    def ones(x):
        c = 0
        p = 1
        while p * 2 <= x:
            p *= 2
        while p >= 1:               # p đi từ lũy thừa lớn nhất của 2 xuống 1
            if x >= p:
                x -= p
                c += 1
            p //= 2
        return c
    s1, s2 = ones(a), ones(b)
    if s1 >= s2:
        return s1 * s2
    # s2 / s1 bằng chia nguyên: đếm số lần trừ được
    q = 0
    rem = s2
    while rem >= s1:
        rem -= s1
        q += 1
    return q


impls = {
    "string": score_string,
    "shift": score_shift,
    "kernighan": score_kernighan,
    "builtin": score_builtin,
    "table": score_table,
    "submit": sol.calculateScore,
}

for a, b, expected in examples:
    assert oracle(a, b) == expected
    for key, fn in impls.items():
        assert fn(a, b) == expected, (key, a, b, fn(a, b), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [17]:
edge_cases = [
    (1, 1, 1, "cả hai bằng 1: s1=s2=1, điểm 1*1"),
    (1, 100, 3, "a=1 (s1=1), b=100=1100100 (s2=3): 3 // 1 = 3"),
    (100, 1, 3, "đảo vai: s1=3 >= s2=1: 3 * 1 = 3"),
    (64, 100, 3, "64 = 1000000 (s1=1), 100 có 3 bit 1 -> 3 // 1 = 3"),
    (100, 99, 1, "100 (3 bit 1) và 99 = 1100011 (4 bit 1): s1=3 < s2=4 -> 4 // 3 = 1 (phần nguyên, không phải 1.33)"),
    (63, 63, 36, "63 = 111111: s1=s2=6 -> 36"),
    (7, 8, 3, "7=111 (3), 8=1000 (1): 3 * 1 = 3"),
    (8, 7, 3, "8 (1), 7 (3): 3 // 1 = 3"),
]
for a, b, expected, label in edge_cases:
    assert oracle(a, b) == expected, (label, oracle(a, b), expected)
    for key, fn in impls.items():
        assert fn(a, b) == expected, (key, label, a, b, fn(a, b), expected)
    print("PASS", label)

PASS cả hai bằng 1: s1=s2=1, điểm 1*1
PASS a=1 (s1=1), b=100=1100100 (s2=3): 3 // 1 = 3
PASS đảo vai: s1=3 >= s2=1: 3 * 1 = 3
PASS 64 = 1000000 (s1=1), 100 có 3 bit 1 -> 3 // 1 = 3
PASS 100 (3 bit 1) và 99 = 1100011 (4 bit 1): s1=3 < s2=4 -> 4 // 3 = 1 (phần nguyên, không phải 1.33)
PASS 63 = 111111: s1=s2=6 -> 36
PASS 7=111 (3), 8=1000 (1): 3 * 1 = 3
PASS 8 (1), 7 (3): 3 // 1 = 3


### 12.2. Tính chất và phản ví dụ

- Với `a == b` luôn có `s1 == s2` nên điểm `= s1 ** 2`.
- Khi `s1 < s2` điểm là thương nguyên `s2 // s1 >= 1`; khi `s1 >= s2` điểm `s1 * s2 >= 1`. Vì `a, b >= 1`, **điểm luôn `>= 1`**.
- **Phản ví dụ** cho lỗi dùng `/`: `a=100 (s1=3), b=99 (s2=4)` cho `4 / 3 = 1.333...` (float) thay vì `1`.

In [18]:
for x in range(1, 101):
    assert sol.calculateScore(x, x) == x.bit_count() ** 2
for a in range(1, 101):
    for b in range(1, 101):
        assert sol.calculateScore(a, b) >= 1
print("PASS tính chất (a == b và điểm >= 1) trên mọi cặp 1..100")


def sai_chia_thuc(a, b):
    s1, s2 = a.bit_count(), b.bit_count()
    return s1 * s2 if s1 >= s2 else s2 / s1          # SAI: / ra float


print("phản ví dụ (100, 99): đúng =", sol.calculateScore(100, 99), "| dùng '/':", sai_chia_thuc(100, 99))
assert sai_chia_thuc(100, 99) != sol.calculateScore(100, 99)

PASS tính chất (a == b và điểm >= 1) trên mọi cặp 1..100
phản ví dụ (100, 99): đúng = 1 | dùng '/': 1.3333333333333333


### 12.3. Đối chiếu toàn bộ và ngẫu nhiên (có seed)

Miền đầu vào chỉ có `100 × 100 = 10 000` cặp, nên ta kiểm tra **vét cạn toàn bộ** mọi cặp, cộng thêm một đợt ngẫu nhiên có seed cho đủ quy trình.

In [19]:
for a in range(1, 101):
    for b in range(1, 101):
        truth = oracle(a, b)
        for key, fn in impls.items():
            assert fn(a, b) == truth, (key, a, b, fn(a, b), truth)
print("PASS vét cạn 10 000 cặp (a, b) trong [1, 100]")

random.seed(4092)
for _ in range(2000):
    a, b = random.randint(1, 100), random.randint(1, 100)
    assert all(fn(a, b) == oracle(a, b) for fn in impls.values())
print("PASS 2000 cặp ngẫu nhiên có seed")

PASS vét cạn 10 000 cặp (a, b) trong [1, 100]


PASS 2000 cặp ngẫu nhiên có seed


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `/` thay vì `//`

In [20]:
def sai_float(a, b):
    s1, s2 = bin(a).count("1"), bin(b).count("1")
    return s1 * s2 if s1 >= s2 else s2 / s1           # SAI


for a, b in [(5, 15), (100, 99)]:
    print((a, b), "sai:", sai_float(a, b), "| đúng:", Solution().calculateScore(a, b))

(5, 15) sai: 2.0 | đúng: 2
(100, 99) sai: 1.3333333333333333 | đúng: 1


**Vì sao sai:** `4 / 2 = 2.0` (float, có thể bị chấm sai kiểu) và `4 / 3 = 1.333...` khác `1`. Đề yêu cầu **chia nguyên**: dùng `//`.

### Lỗi 2: Đếm độ dài nhị phân thay vì số bit `1`
#### Code sai

In [21]:
def sai_do_dai(a, b):
    s1, s2 = len(bin(a)) - 2, len(bin(b)) - 2        # SAI: đây là số chữ số nhị phân
    return s1 * s2 if s1 >= s2 else s2 // s1


print("14, 6  sai:", sai_do_dai(14, 6), "| đúng:", Solution().calculateScore(14, 6))
print("10, 12 sai:", sai_do_dai(10, 12), "| đúng:", Solution().calculateScore(10, 12))

14, 6  sai: 12 | đúng: 6
10, 12 sai: 16 | đúng: 4


**Vì sao sai:** `len(bin(x)) - 2` đếm **số chữ số** của dạng nhị phân (với `10 = 1010` là `4`), còn đề cần **số bit `1`** (là `2`). **Code đúng:** `bin(x).count("1")` hoặc `x.bit_count()`.

### Lỗi 3: Đảo hai nhánh (`>` / `>=`, hoặc `/` ở nhánh nhân)

Nhánh `s1 == s2` phải **nhân** (đề ghi `s1 >= s2`). Nếu bạn viết `if s1 > s2: nhân else: chia` thì khi `s1 == s2` bạn chia ra `1` thay vì `s1 * s2`.
#### Code sai

In [22]:
def sai_nhanh(a, b):
    s1, s2 = a.bit_count(), b.bit_count()
    return s1 * s2 if s1 > s2 else s2 // s1          # SAI: thiếu dấu =


print("10, 12 sai:", sai_nhanh(10, 12), "| đúng:", Solution().calculateScore(10, 12))

10, 12 sai: 1 | đúng: 4


**Vì sao sai:** với `10` và `12`, `s1 = s2 = 2`; đề cho `2 * 2 = 4` nhưng bản sai trả `2 // 2 = 1`. **Code đúng:** dùng `s1 >= s2`.

### Lỗi 4: Đếm bit `0` bằng `bin(x).count('0')` (dính tiền tố `0b`)
#### Code sai

In [23]:
x = 15
print("số chữ số 0 thật của 15 (1111) là 0, nhưng bin(15).count('0') =", bin(15).count("0"))

số chữ số 0 thật của 15 (1111) là 0, nhưng bin(15).count('0') = 1


**Vì sao sai:** chuỗi `'0b1111'` có một ký tự `'0'` ở tiền tố. Bài này đếm bit `1` nên không bị ảnh hưởng; nhưng bài khác cần số bit `0` phải đếm trên `bin(x)[2:]`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "số bit 1", "set bits", "Hamming weight", "popcount" → `x.bit_count()` hoặc `bin(x).count("1")`; muốn tự viết thì **Kernighan** `x &= x - 1`.
- Thấy "integer division" → `//`, luôn kiểm tra mẫu số **khác 0**.
- Đề có công thức "nếu ... thì ... ngược lại ..." → viết **đúng thứ tự và dấu** (`>=` vs `>`), rồi thử ca `s1 == s2`.
- Miền `1..100` nhỏ → có thể **vét cạn mọi cặp** để chắc chắn (như phần 12.3).
- Ghi nhớ vài popcount nhanh: `2^k` chỉ có `1` bit `1`; `2^k - 1` có `k` bit `1`.
- Nhẩm nhanh: `x & 1` là chẵn/lẻ, `x >> 1` là `x // 2`.

## 15. Biến thể của bài

1. Đếm số bit `0` (không tính phần đầu): `x.bit_length() - x.bit_count()`.
2. **Hamming distance** giữa `a` và `b`: `(a ^ b).bit_count()`.
3. Sắp xếp mảng theo số bit `1` (LeetCode 1356).
4. Đếm số bit `1` của mọi số `0..n` (LeetCode 338 — chính là Cách 5).
5. Cho phép `a = 0`: lúc đó phải định nghĩa lại kết quả khi `s1 = 0`.

In [24]:
def zero_bits(x):
    return x.bit_length() - x.bit_count()


def hamming_distance(a, b):
    return (a ^ b).bit_count()


def counting_bits(n):
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)
    return ans


print(zero_bits(10), hamming_distance(1, 4), counting_bits(5))   # 2 2 [0, 1, 1, 2, 1, 2]

2 2 [0, 1, 1, 2, 1, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi `bin` | đổi sang chuỗi rồi đếm `'1'` | `O(B)` | `O(B)` | dễ hiểu nhất, người mới |
| 2. Dịch bit | `x & 1` rồi `x >>= 1` | `O(B)` | `O(1)` | khi cần học toán tử bit cơ bản |
| 3. Kernighan | `x &= x - 1`, đếm số vòng | `O(s)` | `O(1)` | ít bit `1`, hay gặp trong phỏng vấn |
| 4. `bit_count()` | hàm dựng sẵn | `O(1)` | `O(1)` | **bản nộp gọn nhất** (Python 3.10+) |
| 5. Bảng DP | `cnt[i] = cnt[i>>1] + (i&1)` | dựng `O(M)`, tra `O(1)` | `O(M)` | nhiều truy vấn trên miền nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `bin(10)`; (b) `bin(10).count("1")`; (c) `13 & 1` và `13 >> 1`; (d) `7 // 2` và `7 / 2`.

**Bài 2 (tính bằng tay).** Tính điểm của `(a, b) = (7, 9)` và `(a, b) = (4, 31)`; ghi rõ `s1`, `s2`.

**Bài 3 (điền TODO).** Hoàn thiện `popcount_todo(x)` bằng thủ thuật Kernighan.

In [25]:
def popcount_todo(x):
    count = 0
    # TODO 1: lặp khi x còn khác 0
    # TODO 2: xóa bit 1 thấp nhất bằng x &= x - 1
    # TODO 3: tăng count lên 1
    # TODO 4: trả count
    pass


# Tự làm xong rồi mới mở test:
# assert popcount_todo(14) == 3 and popcount_todo(0) == 0 and popcount_todo(100) == 3

**Bài 4 (tự viết lại).** Viết `score_mask(a, b)` đếm bit bằng cách duyệt `for i in range(7): if a >> i & 1: ...` (kiểm tra từng vị trí bit `0..6`) rồi áp công thức.

**Bài 5 (biến thể / mini challenge).** Cho phép `a, b >= 0` (có thể bằng `0`). Định nghĩa: nếu `s1 == 0` thì điểm `= 0`. Viết `score_safe(a, b)` và kiểm tra `score_safe(0, 5) == 0`, `score_safe(5, 0) == 0`, `score_safe(5, 15) == 2`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `bin` có tiền tố `0b`; `13 = 1101`; `//` bỏ phần thập phân. Bài 2: đổi sang nhị phân rồi đếm số `1`. Bài 3: mỗi vòng xóa đúng một bit `1`.
- **Mức 2 (vừa):** Bài 1: `'0b1010'`, `2`, `1` và `6`, `3` và `3.5`. Bài 2: `7 = 111` (3), `9 = 1001` (2); `4 = 100` (1), `31 = 11111` (5). Bài 3: `while x: x &= x - 1; count += 1`. Bài 5: khi `s1 == 0` thì trả `0` sớm (ở **cả** hai nhánh phải tránh chia cho `0`).
- **Mức 3 (sâu):** Bài 2: `(7, 9)`: `s1 = 3 >= s2 = 2` nên điểm `6`; `(4, 31)`: `s1 = 1 < s2 = 5` nên điểm `5 // 1 = 5`. Bài 4: `s = sum(a >> i & 1 for i in range(7))`. Bài 5: `if s1 == 0: return 0`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert bin(10) == "0b1010" and bin(10).count("1") == 2
assert (13 & 1, 13 >> 1) == (1, 6)
assert (7 // 2, 7 / 2) == (3, 3.5)

# Bài 2
assert Solution().calculateScore(7, 9) == 6      # s1=3, s2=2
assert Solution().calculateScore(4, 31) == 5     # s1=1, s2=5 -> 5 // 1


# Bài 3
def popcount_todo_ans(x):
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count


assert popcount_todo_ans(14) == 3 and popcount_todo_ans(0) == 0 and popcount_todo_ans(100) == 3


# Bài 4
def score_mask(a, b):
    s1 = sum(a >> i & 1 for i in range(7))
    s2 = sum(b >> i & 1 for i in range(7))
    return s1 * s2 if s1 >= s2 else s2 // s1


for a, b, expected in examples:
    assert score_mask(a, b) == expected


# Bài 5
def score_safe(a, b):
    s1, s2 = a.bit_count(), b.bit_count()
    if s1 == 0:
        return 0
    return s1 * s2 if s1 >= s2 else s2 // s1


assert score_safe(0, 5) == 0 and score_safe(5, 0) == 0 and score_safe(5, 15) == 2
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (popcount rồi áp công thức)

```python
class Solution:
    def calculateScore(self, a: int, b: int) -> int:
        s1 = a.bit_count()          # hoặc bin(a).count("1")
        s2 = b.bit_count()
        return s1 * s2 if s1 >= s2 else s2 // s1
```

**Checklist**

- [ ] Đếm **số bit `1`** (popcount), không phải độ dài nhị phân?
- [ ] Nhánh `s1 >= s2` (có dấu `=`) → **nhân**; nhánh còn lại → chia **nguyên** `//`?
- [ ] Không chia cho `0` (đã kiểm tra `a, b >= 1` nên `s1 >= 1`)?
- [ ] Đã thử ca `s1 == s2`, `a == b`, `a = 1`, `100`, `63`?
- [ ] Đã vét cạn / đối chiếu với brute force độc lập trên mọi cặp?
- [ ] Nếu máy chấm Python cũ: dùng `bin(x).count("1")` thay `bit_count()`?
- [ ] Skeleton đúng `class Solution` / `def calculateScore(self, a: int, b: int) -> int`?